<a href="https://colab.research.google.com/github/JeanMusenga/SZUniversity/blob/main/Ablating_Association_CPArchISLinker(RQ5_A).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# RQ5-L: CPArchISLinker without Lexical Features (Association feature A is zeroed out)
# ==========================================================

!pip install -q sentence-transformers rapidfuzz openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random, re
from sentence_transformers import SentenceTransformer
from rapidfuzz import fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
import matplotlib.pyplot as plt

# ------------------ Reproducibility ------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
# ==========================================================
# Load Dataset
# ==========================================================
PATH = "Dataset_GitHub_Solutions.xlsx"
df = pd.read_excel(PATH)
required = {'commit_id','so_id','p_text','s_text','label'}
assert required.issubset(df.columns)
df['label'] = df['label'].astype(int)

# ------------------ Commit-level 80/20 split ------------------
commit_ids = df.commit_id.unique()
train_cids, test_cids = train_test_split(commit_ids, test_size=0.2, random_state=SEED)

df_train = df[df.commit_id.isin(train_cids)].copy()
df_test  = df[df.commit_id.isin(test_cids)].copy()

print("Train commits:", len(train_cids))
print("Test commits :", len(test_cids))

Train commits: 786
Test commits : 197


# I. Preprocessing Layer

In [ ]:
# ==========================================================
# I. Preprocessing
# ==========================================================
def clean_text(s):
    if pd.isna(s): return ""
    s = str(s).lower()
    s = re.sub(r"`{3}.*?`{3}", " ", s, flags=re.S)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"&[a-zA-Z]+;", " ", s)
    s = re.sub(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+",
        '',
        s,
        flags=re.UNICODE
    )
    s = re.sub(r"\s+", " ", s)
    return s.strip()

for d in [df_train, df_test]:
    d["p_text"] = d["p_text"].apply(clean_text)
    d["s_text"] = d["s_text"].apply(clean_text)

# II. Feature Extraction

In [ ]:
# ==========================================================
# II. Feature Extraction (CPArchISLinker – Refactored)
# ==========================================================

sbert = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# ---------- Embedding Cache ----------
_EMB = {}

def embed(text: str) -> np.ndarray:
    text = text.strip().lower()
    if text not in _EMB:
        _EMB[text] = sbert.encode(
            text,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype(np.float32)
    return _EMB[text]

# ---------- Precompute Commit Keywords ----------
commit_kw = {
    cid: re.findall(r"\b\w+\b", txt)
    for cid, txt in zip(df.commit_id, df.p_text)
}

# ---------- Lexical Feature (L) ----------
def lexical_score(commit_id, s_text):
    kws = commit_kw.get(commit_id, [])
    if not kws:
        return 0.0
    return np.mean([fuzz.partial_ratio(k, s_text) / 100 for k in kws])

# ---------- Semantic Similarity (S) ----------
def semantic_score(vp, vs):
    return cosine_similarity(vp.reshape(1, -1), vs.reshape(1, -1))[0, 0]

# ---------- Association Rules (A) ----------
PROBLEM_TERMS = {
    "performance": ["latency", "slow", "bottleneck", "throughput", "performance", "latency"],
    "scalability": ["scale", "scaling", "load", "traffic", "scalability",  "high traffic"],
    "security": ["authentication", "authorization", "vulnerability", "security", "authentication", "encryption", "vulnerability"],
    "maintainability": ["refactor", "technical debt", "maintainability",  "clean code"],
    "reliability": ["crash", "failure", "availability", "reliability", "fault"],
    "deployment": ["deployment", "docker", "kubernetes", "ci/cd"],
    "communication": ["communication", "rpc", "rest", "http", "message"],
    "anti_pattern": ["god class", "spaghetti", "anti-pattern"],
}

SOLUTION_TERMS = {
    "architectural_pattern": ["microservice", "mvc", "layered", "event-driven"],
    "architectural_tactic": ["caching", "load balancing", "replication"],
    "framework": ["spring", "django", "react", "flask"],
    "api": ["api", "rest", "grpc"],
    "protocol": ["http", "https", "tcp"],
    "refactoring": ["refactor", "restructure", "redesign"],
}

ACTION_VERBS = {
    "optimize", "refactor", "migrate", "replace",
    "remove", "add", "improve", "reduce", "fix"
}

def contains_any(text, terms):
    return any(t in text for t in terms)

def extract_actions(text):
    return {v for v in ACTION_VERBS if v in text}

def association_score(p_text, s_text):
    p = p_text.lower()
    s = s_text.lower()

    rule_1 = any(
        contains_any(p, v) and contains_any(s, v)
        for v in PROBLEM_TERMS.values()
    )

    rule_2 = any(
        contains_any(p, v) and contains_any(s, v)
        for v in SOLUTION_TERMS.values()
    )

    rule_3 = bool(extract_actions(p) & extract_actions(s))

    return np.mean([rule_1, rule_2, rule_3])

# ---------- MCI ----------
def compute_mci(L, S, A, α=0.3, β=0.4, γ=0.3):
    return α * L + β * S + γ * A



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def build_features(row, drop=None):
    """
    drop ∈ {None, 'lexical', 'semantic', 'association'}
    """
    vp = embed(row.p_text)
    vs = embed(row.s_text)

    L = lexical_score(row.commit_id, row.s_text)
    S = semantic_score(vp, vs)
    A = association_score(row.p_text, row.s_text)

    if drop == "lexical":
        L = 0.0
    elif drop == "semantic":
        S = 0.0
    elif drop == "association":
        A = 0.0

    MCI = compute_mci(L, S, A)

    return np.array([L, S, A, MCI], dtype=np.float32)


In [ ]:
def build_matrix(df, drop=None):
    X = np.vstack(df.apply(build_features, axis=1, drop=drop))
    y = df.label.values.astype(np.float32)
    return X, y


## Train/Test Split

In [ ]:
# ==========================================================
# RQ5-L: CPArchISLinker without Lexical Features
# (Lexical feature L is zeroed out)
# ==========================================================

print("\n" + "="*70)
print("Running CPArchISLinker-Lexical (RQ5-L)")
print("="*70)

# ----------------------------------------------------------
# 1. Feature Extraction (Lexical Ablated)
# ----------------------------------------------------------
X_train_raw, y_train = build_matrix(df_train, drop="association")
X_test_raw,  y_test  = build_matrix(df_test,  drop="association")

# ----------------------------------------------------------
# 2. Scaling (re-fit per variant to avoid leakage)
# ----------------------------------------------------------
scaler = StandardScaler().fit(X_train_raw)

X_train = scaler.transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

# ----------------------------------------------------------
# 3. Tensor Conversion
# ----------------------------------------------------------
X_train_t = torch.tensor(X_train, device=device)
X_test_t  = torch.tensor(X_test, device=device)

y_train_t = torch.tensor(y_train, device=device)
y_test_t  = torch.tensor(y_test, device=device)



Running CPArchISLinker-Lexical (RQ5-L)


# III Relevancy Identification Layer

In [ ]:
# ==================================================
# Stage 1 — DMLwith Hard Negative Mining (RESTORED)
# ==================================================
class DML_Stage1(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(d,256), nn.ReLU(),
            nn.Linear(256,128), nn.ReLU(),
            nn.Linear(128,64)
        )
        self.cls = nn.Sequential(
            nn.Linear(64,32), nn.ReLU(),
            nn.Linear(32,1)
        )

    def forward(self, x):
        h = self.embed(x)
        h = h / (h.norm(dim=1, keepdim=True) + 1e-9)
        return self.cls(h).squeeze(), h


net = DML_Stage1(X_train.shape[1]).to(device)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()


# ---------- Hard Negative Mining ----------
def mine_hard_negatives(df_split, H):
    pairs = []
    idx_map = {idx: i for i, idx in enumerate(df_split.index)}

    for cid, g in df_split.groupby("commit_id"):
        pos = g[g.label == 1].index
        neg = g[g.label == 0].index

        for p in pos:
            sims = [np.dot(H[idx_map[p]], H[idx_map[n]]) for n in neg]
            if sims:
                hard_n = neg[np.argmax(sims)]
                pairs.append((p, hard_n))
    return pairs


## Stage-1 Training -

In [ ]:
# ----------------------------------------------------------
# 4. Stage-1 Training (DML + Hard Negative Mining)
# ----------------------------------------------------------
net = DML_Stage1(X_train.shape[1]).to(device)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

EPOCHS_STAGE1 = 60

for epoch in range(EPOCHS_STAGE1):
    net.train()

    with torch.no_grad():
        _, H = net(X_train_t)
        H = H.cpu().numpy()

    pairs = mine_hard_negatives(df_train, H)
    random.shuffle(pairs)

    total_loss = 0.0

    for p, n in pairs:
        Xp = X_train_t[df_train.index.get_loc(p)]
        Xn = X_train_t[df_train.index.get_loc(n)]

        xb = torch.stack([Xp, Xn])
        yb = torch.tensor([1.0, 0.0], device=device)

        opt.zero_grad()
        logits, _ = net(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f"[RQ5-L | Stage-1] Epoch {epoch+1:02d}/{EPOCHS_STAGE1} | Loss {total_loss/len(pairs):.4f}")


[RQ5-L | Stage-1] Epoch 01/60 | Loss 0.1870
[RQ5-L | Stage-1] Epoch 02/60 | Loss 0.1172
[RQ5-L | Stage-1] Epoch 03/60 | Loss 0.1156
[RQ5-L | Stage-1] Epoch 04/60 | Loss 0.1155
[RQ5-L | Stage-1] Epoch 05/60 | Loss 0.1096
[RQ5-L | Stage-1] Epoch 06/60 | Loss 0.1074
[RQ5-L | Stage-1] Epoch 07/60 | Loss 0.1117
[RQ5-L | Stage-1] Epoch 08/60 | Loss 0.1087
[RQ5-L | Stage-1] Epoch 09/60 | Loss 0.1053
[RQ5-L | Stage-1] Epoch 10/60 | Loss 0.1069
[RQ5-L | Stage-1] Epoch 11/60 | Loss 0.1059
[RQ5-L | Stage-1] Epoch 12/60 | Loss 0.1069
[RQ5-L | Stage-1] Epoch 13/60 | Loss 0.1050
[RQ5-L | Stage-1] Epoch 14/60 | Loss 0.1023
[RQ5-L | Stage-1] Epoch 15/60 | Loss 0.1034
[RQ5-L | Stage-1] Epoch 16/60 | Loss 0.1041
[RQ5-L | Stage-1] Epoch 17/60 | Loss 0.1045
[RQ5-L | Stage-1] Epoch 18/60 | Loss 0.1036
[RQ5-L | Stage-1] Epoch 19/60 | Loss 0.1043
[RQ5-L | Stage-1] Epoch 20/60 | Loss 0.1026
[RQ5-L | Stage-1] Epoch 21/60 | Loss 0.1019
[RQ5-L | Stage-1] Epoch 22/60 | Loss 0.1015
[RQ5-L | Stage-1] Epoch 23/60 | 

## Stage-1 Evaluation

In [ ]:
# ---------- Stage-1 Evaluation ----------
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

net.eval()
with torch.no_grad():
    # Train set (for Stage-2 usage, not reporting)
    logits_train, _ = net(X_train_t)
    probs_train = torch.sigmoid(logits_train).cpu().numpy()

    # Test set (for reporting)
    logits_test, _ = net(X_test_t)
    probs_test = torch.sigmoid(logits_test).cpu().numpy()
    preds_test = (probs_test >= 0.5).astype(int)

y_test_np = y_test_t.cpu().numpy()

print("\n" + "=" * 60)
print("Stage-1 Evaluation Results (Binary Classification)")
print("=" * 60)

print("\nClassification Report:")
print(classification_report(y_test_np, preds_test, digits=4))

print("ROC-AUC:", roc_auc_score(y_test_np, probs_test))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_np, preds_test))



Stage-1 Evaluation Results (Binary Classification)

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9694    0.9645    0.9669       197
         1.0     0.9650    0.9698    0.9674       199

    accuracy                         0.9672       396
   macro avg     0.9672    0.9672    0.9672       396
weighted avg     0.9672    0.9672    0.9672       396

ROC-AUC: 0.9926026069433461

Confusion Matrix:
[[190   7]
 [  6 193]]


In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

p, r, f, _ = precision_recall_fscore_support(
    y_test_np, preds_test, average="binary"
)
auc = roc_auc_score(y_test_np, probs_test)

print({"P": p, "R": r, "F1": f, "AUC": auc})


{'P': 0.965, 'R': 0.9698492462311558, 'F1': 0.9674185463659147, 'AUC': np.float64(0.9926026069433461)}


# IV. Linking and Recommendation Layer

In [ ]:
# ==========================================================
# 1. Extract Stage-1 latent representations
# ==========================================================
net.eval()
with torch.no_grad():
    _, H_train = net(X_train_t)
    _, H_test  = net(X_test_t)

H_train = H_train.cpu().numpy()
H_test  = H_test.cpu().numpy()

# ==========================================================
# 2. Cross-commit similarity-based retrieval
# ==========================================================
def retrieve_cross_commit_indices(anchor_idx, df_split, H_split, top_m=30):
    sims = cosine_similarity(
        H_split[anchor_idx].reshape(1, -1),
        H_split
    ).flatten()

    anchor_cid = df_split.iloc[anchor_idx].commit_id

    tmp = df_split.copy()
    tmp["sim"] = sims
    tmp["pos"] = range(len(tmp))

    tmp = tmp[tmp.commit_id != anchor_cid]
    tmp = tmp.sort_values("sim", ascending=False)

    return tmp.head(top_m)["pos"].tolist()

# ==========================================================
# 3. Build Stage-2 candidates (OWN + CROSS-COMMIT)
# ==========================================================
def build_stage2_candidates(df_split, X_raw, probs, H_split, top_m=30):
    all_feats = np.array([
        np.concatenate([X_raw[i], [probs[i]]])
        for i in range(len(df_split))
    ])

    candidates = {}

    for cid, g in df_split.groupby("commit_id"):
        own_pos = [df_split.index.get_loc(i) for i in g.index]
        anchor_idx = own_pos[0]

        cross_pos = retrieve_cross_commit_indices(
            anchor_idx, df_split, H_split, top_m
        )

        merged_pos = list(set(own_pos + cross_pos))
        candidates[cid] = merged_pos

    return candidates, all_feats

# ==========================================================
# 4. Build pairwise training data
# ==========================================================
def build_stage2_pairs(df_train, candidates, all_feats):
    pairs = []

    for cid, pos_list in candidates.items():
        g = df_train[df_train.commit_id == cid]

        pos_feats = [
            all_feats[df_train.index.get_loc(i)]
            for i in g[g.label == 1].index
        ]

        neg_feats = [
            all_feats[p]
            for p in pos_list
            if not any(np.array_equal(all_feats[p], pf) for pf in pos_feats)
        ]

        for fp in pos_feats:
            for fn in neg_feats:
                pairs.append((fp, fn))

    return pairs

# ==========================================================
# 5. Prepare training data
# ==========================================================
train_candidates, train_all_feats = build_stage2_candidates(
    df_train, X_train_raw, probs_train, H_train, top_m=30
)

train_pairs = build_stage2_pairs(
    df_train, train_candidates, train_all_feats
)

print("Stage-2 training pairs:", len(train_pairs))

# ==========================================================
# 6. RankNet model
# ==========================================================
class RankNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze()

ranker = RankNet(train_pairs[0][0].shape[0]).to(device)
opt2 = torch.optim.Adam(ranker.parameters(), lr=1e-3)

def pairwise_rank_loss(sp, sn, margin=1.0):
    return torch.mean(torch.clamp(margin - (sp - sn), min=0))


Stage-2 training pairs: 25244


## Stage-2 Training

In [ ]:
# ==========================================================
# 7. Stage-2 Training
# ==========================================================
for epoch in range(40):
    random.shuffle(train_pairs)
    total = 0.0

    for fp, fn in train_pairs:
        fp = torch.tensor(fp, device=device)
        fn = torch.tensor(fn, device=device)

        loss = pairwise_rank_loss(ranker(fp), ranker(fn))
        opt2.zero_grad()
        loss.backward()
        opt2.step()

        total += loss.item()

    print(f"[Stage-2] Epoch {epoch+1:02d} | Loss {total/len(train_pairs):.4f}")

[Stage-2] Epoch 01 | Loss 0.9624
[Stage-2] Epoch 02 | Loss 0.9573
[Stage-2] Epoch 03 | Loss 0.9546
[Stage-2] Epoch 04 | Loss 0.9511
[Stage-2] Epoch 05 | Loss 0.9480
[Stage-2] Epoch 06 | Loss 0.9441
[Stage-2] Epoch 07 | Loss 0.9422
[Stage-2] Epoch 08 | Loss 0.9474
[Stage-2] Epoch 09 | Loss 0.9426
[Stage-2] Epoch 10 | Loss 0.9424
[Stage-2] Epoch 11 | Loss 0.9400
[Stage-2] Epoch 12 | Loss 0.9421
[Stage-2] Epoch 13 | Loss 0.9405
[Stage-2] Epoch 14 | Loss 0.9404
[Stage-2] Epoch 15 | Loss 0.9405
[Stage-2] Epoch 16 | Loss 0.9429
[Stage-2] Epoch 17 | Loss 0.9453
[Stage-2] Epoch 18 | Loss 0.9440
[Stage-2] Epoch 19 | Loss 0.9438
[Stage-2] Epoch 20 | Loss 0.9420
[Stage-2] Epoch 21 | Loss 0.9427
[Stage-2] Epoch 22 | Loss 0.9419
[Stage-2] Epoch 23 | Loss 0.9464
[Stage-2] Epoch 24 | Loss 0.9431
[Stage-2] Epoch 25 | Loss 0.9419
[Stage-2] Epoch 26 | Loss 0.9412
[Stage-2] Epoch 27 | Loss 0.9422
[Stage-2] Epoch 28 | Loss 0.9418
[Stage-2] Epoch 29 | Loss 0.9420
[Stage-2] Epoch 30 | Loss 0.9411
[Stage-2] 

## Stage-2 Evaluation

In [ ]:
# ==========================================================
# 8. Stage-2 Evaluation (Cross-Commit P@K & MRR)
# ==========================================================
def evaluate_stage2(df_split, X_raw, probs, H_split, ks=[1,3,5,10]):
    candidates, all_feats = build_stage2_candidates(
        df_split, X_raw, probs, H_split, top_m=30
    )

    hr = {k: 0 for k in ks}
    mrr = []

    for cid, pos_list in candidates.items():
        scores, labels = [], []

        for p in pos_list:
            with torch.no_grad():
                s = ranker(
                    torch.tensor(all_feats[p], device=device)
                ).item()
            scores.append(s)
            labels.append(df_split.iloc[p].label)

        order = np.argsort(scores)[::-1]
        labels = np.array(labels)[order]

        pos = np.where(labels == 1)[0]
        mrr.append(1 / (pos[0] + 1) if len(pos) > 0 else 0)

        for k in ks:
            hr[k] += int(labels[:k].sum() > 0)

    n = df_split.commit_id.nunique()
    return {f"HR@{k}": hr[k] / n for k in ks}, np.mean(mrr)

hr, mrr = evaluate_stage2(
    df_test, X_test_raw, probs_test, H_test
)

print("\nStage-2 Cross-Commit Results")
print("HR:", hr)
print("MRR:", mrr)


Stage-2 Cross-Commit Results
HR: {'HR@1': 0.9847715736040609, 'HR@3': 0.9898477157360406, 'HR@5': 0.9898477157360406, 'HR@10': 0.9898477157360406}
MRR: 0.9880312531103811


In [ ]:
# ==========================================================
# 9. Save Cross-Commit Top-K Results
# ==========================================================
top_k = 10

test_candidates, test_all_feats = build_stage2_candidates(
    df_test, X_test_raw, probs_test, H_test, top_m=30
)

rows = []

for cid, pos_list in test_candidates.items():
    scores = []

    for p in pos_list:
        with torch.no_grad():
            s = ranker(
                torch.tensor(test_all_feats[p], device=device)
            ).item()
        scores.append(s)

    order = np.argsort(scores)[::-1][:top_k]

    for r, idx in enumerate(order, 1):
        row = df_test.iloc[pos_list[idx]].copy()
        row["stage2_score"] = scores[idx]
        row["stage2_rank"] = r
        rows.append(row)

df_stage2_topk = pd.DataFrame(rows)
df_stage2_topk.to_excel(
    "Association_Ablated_Stage2_CrossCommit_TopK_Ranked_Results.xlsx",
    index=False
)

print("\nSaved: Association_Ablated_Stage2_CrossCommit_TopK_Ranked_Results.xlsx")



Saved: Association_Ablated_Stage2_CrossCommit_TopK_Ranked_Results.xlsx
